# 4. Train/Test Construction

Subject-level train/test split을 만들고, 이후 전처리와 모델링에서 재사용할 LSTM sequence index를 생성합니다.

- 같은 `subject_id`는 반드시 하나의 split에만 속합니다.
- split은 가능하면 subject-level `ever_delirium` 기준으로 stratified random split을 수행합니다.
- 12시간 bin-level LSTM 입력은 4개 input step과 3개 target step(`t`, `t+1`, `t+2`)을 기준으로 구성합니다.


## 1. 환경 설정

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
TEST_SIZE = 0.20
INPUT_STEPS = 4
TARGET_STEPS = 3

SRC_DIR = Path.cwd().resolve()
PROJECT_DIR = SRC_DIR.parent
TRANSFORM_DIR = PROJECT_DIR / "processed" / "transform"
MODELING_DIR = PROJECT_DIR / "processed" / "modeling"

MODELING_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"TRANSFORM_DIR: {TRANSFORM_DIR}")
print(f"MODELING_DIR: {MODELING_DIR}")

## 2. Transform 산출물 로드

In [ ]:
input_files = {
    "binned": TRANSFORM_DIR / "events_12h_binned.csv",
    "wide_charttime": TRANSFORM_DIR / "events_12h_wide_by_charttime.csv",
    "long": TRANSFORM_DIR / "events_12h_long.csv",
    "cohort": TRANSFORM_DIR / "cohort_final.csv",
}

missing_files = [str(path) for path in input_files.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError("Missing required transform outputs:\n" + "\n".join(missing_files))

events_binned = pd.read_csv(input_files["binned"])
events_wide = pd.read_csv(input_files["wide_charttime"])
events_long = pd.read_csv(input_files["long"])
cohort = pd.read_csv(input_files["cohort"])

assessment_path = TRANSFORM_DIR / "assessment_index_12h.csv"
assessment_index = pd.read_csv(assessment_path) if assessment_path.exists() else None

print("events_binned:", events_binned.shape)
print("events_wide:", events_wide.shape)
print("events_long:", events_long.shape)
print("cohort:", cohort.shape)
print("assessment_index:", None if assessment_index is None else assessment_index.shape)

## 3. 기본 QA

In [ ]:
required_binned_cols = {"subject_id", "hadm_id", "stay_id", "bin", "delirium"}
required_cohort_cols = {"subject_id", "hadm_id", "stay_id", "ever_delirium"}

missing_binned_cols = sorted(required_binned_cols - set(events_binned.columns))
missing_cohort_cols = sorted(required_cohort_cols - set(cohort.columns))

if missing_binned_cols:
    raise ValueError(f"events_12h_binned.csv missing columns: {missing_binned_cols}")
if missing_cohort_cols:
    raise ValueError(f"cohort_final.csv missing columns: {missing_cohort_cols}")

for df in [events_binned, events_wide, events_long, cohort]:
    if "split" in df.columns:
        df.drop(columns="split", inplace=True)

events_binned["bin"] = pd.to_numeric(events_binned["bin"], errors="raise").astype(int)
events_binned["delirium"] = pd.to_numeric(events_binned["delirium"], errors="coerce")
cohort["ever_delirium"] = pd.to_numeric(cohort["ever_delirium"], errors="coerce").fillna(0).astype(int)

print("n_subjects:", cohort["subject_id"].nunique())
print("n_stays:", cohort["stay_id"].nunique())
print("n_bins:", len(events_binned))
print("\nSubject-level ever_delirium distribution")
display(cohort.drop_duplicates("subject_id")["ever_delirium"].value_counts(dropna=False).sort_index())
print("\n12h bin-level delirium distribution")
display(events_binned["delirium"].value_counts(dropna=False).sort_index())

## 4. 환자 단위 train/test 분할

In [ ]:
def make_subject_summary(cohort_df: pd.DataFrame) -> pd.DataFrame:
    summary = (
        cohort_df.groupby("subject_id", as_index=False)
        .agg(
            ever_delirium=("ever_delirium", "max"),
            n_stays=("stay_id", "nunique"),
        )
        .sort_values("subject_id")
        .reset_index(drop=True)
    )
    summary["ever_delirium"] = summary["ever_delirium"].fillna(0).astype(int)
    return summary


def subject_level_split(
    subject_summary: pd.DataFrame,
    test_size: float = 0.2,
    random_state: int = 42,
    stratify_col: str = "ever_delirium",
) -> tuple[set, set, str]:
    rng = np.random.default_rng(random_state)
    subjects = subject_summary["subject_id"].to_numpy()
    labels = subject_summary[stratify_col]
    class_counts = labels.value_counts(dropna=False)

    can_stratify = labels.nunique(dropna=False) > 1 and class_counts.min() >= 2
    test_subjects = []

    if can_stratify:
        split_method = f"stratified_by_{stratify_col}"
        for _, group in subject_summary.groupby(stratify_col, dropna=False):
            group_subjects = group["subject_id"].to_numpy()
            n_test = int(round(len(group_subjects) * test_size))
            n_test = min(max(n_test, 1), len(group_subjects) - 1)
            test_subjects.extend(rng.choice(group_subjects, size=n_test, replace=False).tolist())
    else:
        split_method = "random_subject_split"
        n_test = int(round(len(subjects) * test_size))
        n_test = min(max(n_test, 1), len(subjects) - 1)
        test_subjects = rng.choice(subjects, size=n_test, replace=False).tolist()

    test_subjects = set(test_subjects)
    train_subjects = set(subjects) - test_subjects
    return train_subjects, test_subjects, split_method

In [ ]:
subject_summary = make_subject_summary(cohort)
train_subjects, test_subjects, split_method = subject_level_split(
    subject_summary,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify_col="ever_delirium",
)

overlap = train_subjects & test_subjects
if overlap:
    raise AssertionError(f"Subject leakage detected: {len(overlap)} overlapping subjects")

subject_summary["split"] = np.where(subject_summary["subject_id"].isin(train_subjects), "train", "test")

print("split_method:", split_method)
print("train subjects:", len(train_subjects))
print("test subjects:", len(test_subjects))
print("subject overlap:", len(overlap))
display(pd.crosstab(subject_summary["split"], subject_summary["ever_delirium"], margins=True))

## 5. split 컬럼 부여

In [ ]:
split_map = subject_summary.set_index("subject_id")["split"]

def attach_split(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out = df.copy()
    out["split"] = out["subject_id"].map(split_map)
    missing_split = out["split"].isna().sum()
    if missing_split:
        raise ValueError(f"{name}: {missing_split:,} rows could not be assigned to train/test split")
    return out

events_binned_split = attach_split(events_binned, "events_binned")
events_wide_split = attach_split(events_wide, "events_wide")
events_long_split = attach_split(events_long, "events_long")
cohort_split = attach_split(cohort, "cohort")

if assessment_index is not None:
    assessment_index_split = attach_split(assessment_index, "assessment_index")
else:
    assessment_index_split = None

print(events_binned_split["split"].value_counts())
print(cohort_split["split"].value_counts())

## 6. Split QA

In [ ]:
def split_counts(df: pd.DataFrame, label_col: str | None = None) -> pd.DataFrame:
    rows = []
    for split, group in df.groupby("split"):
        row = {
            "split": split,
            "n_rows": len(group),
            "n_subjects": group["subject_id"].nunique(),
            "n_stays": group["stay_id"].nunique() if "stay_id" in group.columns else np.nan,
        }
        if label_col and label_col in group.columns:
            label = pd.to_numeric(group[label_col], errors="coerce")
            row[f"{label_col}_positive"] = int((label == 1).sum())
            row[f"{label_col}_negative"] = int((label == 0).sum())
            row[f"{label_col}_missing"] = int(label.isna().sum())
            row[f"{label_col}_positive_rate"] = (label == 1).mean()
        rows.append(row)
    return pd.DataFrame(rows)

cohort_split_qa = split_counts(cohort_split, "ever_delirium")
binned_split_qa = split_counts(events_binned_split, "delirium")

display(cohort_split_qa)
display(binned_split_qa)

train_stays = set(cohort_split.loc[cohort_split["split"] == "train", "stay_id"])
test_stays = set(cohort_split.loc[cohort_split["split"] == "test", "stay_id"])
stay_overlap = train_stays & test_stays
if stay_overlap:
    raise AssertionError(f"Stay leakage detected: {len(stay_overlap)} overlapping stays")

print("subject overlap:", len(train_subjects & test_subjects))
print("stay overlap:", len(stay_overlap))

## 7. LSTM sequence index 생성

In [ ]:
def build_lstm_sequence_index(
    binned_df: pd.DataFrame,
    input_steps: int = 4,
    target_steps: int = 3,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    records = []
    dropped = []
    required_cols = ["subject_id", "hadm_id", "stay_id", "split", "bin", "delirium"]
    missing_cols = sorted(set(required_cols) - set(binned_df.columns))
    if missing_cols:
        raise ValueError(f"Missing columns for sequence construction: {missing_cols}")

    sort_cols = ["subject_id", "hadm_id", "stay_id", "bin"]
    for stay_id, stay_df in binned_df.sort_values(sort_cols).groupby("stay_id", sort=False):
        stay_df = stay_df.drop_duplicates("bin", keep="first").sort_values("bin").reset_index(drop=True)
        bins = stay_df["bin"].to_numpy(dtype=int)
        labels = pd.to_numeric(stay_df["delirium"], errors="coerce").to_numpy()
        n_candidate = max(0, len(stay_df) - input_steps - target_steps + 2)
        n_kept = 0
        n_nonconsecutive = 0
        n_missing_target = 0

        for anchor_pos in range(input_steps - 1, len(stay_df) - target_steps + 1):
            input_pos = np.arange(anchor_pos - input_steps + 1, anchor_pos + 1)
            target_pos = np.arange(anchor_pos, anchor_pos + target_steps)
            window_pos = np.arange(input_pos[0], target_pos[-1] + 1)
            window_bins = bins[window_pos]

            if not np.array_equal(np.diff(window_bins), np.ones(len(window_bins) - 1, dtype=int)):
                n_nonconsecutive += 1
                continue

            target_labels = labels[target_pos]
            if pd.isna(target_labels).any():
                n_missing_target += 1
                continue

            anchor = stay_df.iloc[anchor_pos]
            record = {
                "example_id": f"{int(stay_id)}_{int(bins[anchor_pos])}",
                "subject_id": anchor["subject_id"],
                "hadm_id": anchor["hadm_id"],
                "stay_id": anchor["stay_id"],
                "split": anchor["split"],
                "anchor_bin": int(bins[anchor_pos]),
                "input_start_bin": int(bins[input_pos[0]]),
                "input_end_bin": int(bins[input_pos[-1]]),
                "target_start_bin": int(bins[target_pos[0]]),
                "target_end_bin": int(bins[target_pos[-1]]),
                "input_bins": ",".join(map(str, bins[input_pos].tolist())),
                "target_bins": ",".join(map(str, bins[target_pos].tolist())),
            }
            for offset, label in enumerate(target_labels):
                label_col = "y_t" if offset == 0 else f"y_t_plus_{offset}"
                record[label_col] = int(label)
            record["y_any_t_to_t_plus_2"] = int(np.nanmax(target_labels) == 1)
            records.append(record)
            n_kept += 1

        dropped.append(
            {
                "stay_id": stay_id,
                "n_bins": len(stay_df),
                "candidate_sequences": n_candidate,
                "kept_sequences": n_kept,
                "dropped_nonconsecutive_bins": n_nonconsecutive,
                "dropped_missing_target_label": n_missing_target,
            }
        )

    sequence_index = pd.DataFrame(records)
    dropped_summary = pd.DataFrame(dropped)
    return sequence_index, dropped_summary

In [ ]:
sequence_index, sequence_drop_summary = build_lstm_sequence_index(
    events_binned_split,
    input_steps=INPUT_STEPS,
    target_steps=TARGET_STEPS,
)

if sequence_index.empty:
    raise ValueError("No LSTM sequences were generated. Check bin continuity and delirium target labels.")

sequence_train = sequence_index[sequence_index["split"] == "train"].reset_index(drop=True)
sequence_test = sequence_index[sequence_index["split"] == "test"].reset_index(drop=True)

print("sequence_index:", sequence_index.shape)
display(sequence_index["split"].value_counts())
display(pd.crosstab(sequence_index["split"], sequence_index["y_any_t_to_t_plus_2"], margins=True))
display(sequence_drop_summary.describe(include="all"))

## 8. 산출물 저장

In [ ]:
train_subject_ids = pd.DataFrame({"subject_id": sorted(train_subjects)})
test_subject_ids = pd.DataFrame({"subject_id": sorted(test_subjects)})

split_summary = pd.concat(
    [
        cohort_split_qa.assign(table="cohort_final"),
        binned_split_qa.assign(table="events_12h_binned"),
        sequence_index.groupby("split", as_index=False)
        .agg(
            n_rows=("example_id", "count"),
            n_subjects=("subject_id", "nunique"),
            n_stays=("stay_id", "nunique"),
            y_positive=("y_any_t_to_t_plus_2", "sum"),
            y_positive_rate=("y_any_t_to_t_plus_2", "mean"),
        )
        .assign(table="lstm_sequence_index"),
    ],
    ignore_index=True,
    sort=False,
)

events_binned_split.to_csv(MODELING_DIR / "events_12h_binned_with_split.csv", index=False)
events_wide_split.to_csv(MODELING_DIR / "events_12h_wide_by_charttime_with_split.csv", index=False)
events_long_split.to_csv(MODELING_DIR / "events_12h_long_with_split.csv", index=False)
cohort_split.to_csv(MODELING_DIR / "cohort_final_with_split.csv", index=False)

if assessment_index_split is not None:
    assessment_index_split.to_csv(MODELING_DIR / "assessment_index_12h_with_split.csv", index=False)

train_subject_ids.to_csv(MODELING_DIR / "train_subject_ids.csv", index=False)
test_subject_ids.to_csv(MODELING_DIR / "test_subject_ids.csv", index=False)
subject_summary.to_csv(MODELING_DIR / "subject_split_summary.csv", index=False)
split_summary.to_csv(MODELING_DIR / "train_test_split_summary.csv", index=False)

sequence_index.to_csv(MODELING_DIR / "lstm_sequence_index.csv", index=False)
sequence_train.to_csv(MODELING_DIR / "lstm_sequence_index_train.csv", index=False)
sequence_test.to_csv(MODELING_DIR / "lstm_sequence_index_test.csv", index=False)
sequence_drop_summary.to_csv(MODELING_DIR / "lstm_sequence_drop_summary.csv", index=False)

print("Saved outputs to", MODELING_DIR)
display(split_summary)

## 다음 단계

`5_data_preprocessing.ipynb`에서는 이 노트북의 split과 sequence index를 기준으로 train set에서 feature 제외 기준, categorical encoding, imputation/scaling 값을 fit하고 test set에는 transform만 적용합니다.